In [1]:
from capture_utils_v2 import CaptureSystem

# Initialize the capture system
system = CaptureSystem()


Pixel format set to RGB8
IC4 Grabber and Sink initialized.


In [ ]:
system.display_drawer()
system.run_aruco_detector()


In [ ]:
system.photometric_calibration()

100%|██████████| 33/33 [00:53<00:00,  1.62s/it]


In [ ]:
# save system orig_proj_corners and corners_img_proj to a file
import pickle
with open("capture_system_state.pkl", "wb") as f:
    pickle.dump((system.orig_proj_corners, system.corners_img_proj, system.orig_img), f)

## Infer

In [ ]:
from capture_utils_v2 import CaptureSystem
import cv2
import pickle

# Initialize the capture system
system = CaptureSystem()


IC4 already opened, using existing grabber and sink.


In [ ]:
image_to_project =  cv2.imread('./results/best_patch_str_16x16_6_2025-12-23_10_34.png')
image_to_project = cv2.cvtColor(image_to_project, cv2.COLOR_BGR2RGB)

In [ ]:
with open("capture_system_state.pkl", "rb") as f:
    system.orig_proj_corners, system.corners_img_proj, system.orig_img = pickle.load(f)

In [ ]:
system.plot_on_screen(image_to_project)

In [ ]:
import torch
# from classfier import predict_raw, weights
# from classfier_ensemble import predict_raw, weights
from classfier_ensemble_v2 import predict_raw, weights
# from classfier_dino import predict_raw, weights

tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()
cap = system.cap

caps = []
results = []
for i in range(2400):
    r = cap.read()[1]
    tr = tt(r)
    with torch.no_grad():
        p = predict_raw(tr.unsqueeze(0).cuda())
        res = weights.meta["categories"][p[0].argmax(0).item()]
        prob = p[0].max(0).values.item() * 100
    
    # add text
    cv2.putText(r, f'Pred: {res}: {prob:.2f}%', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow('frame', r)
    results.append(res)

    caps.append(r)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

Loading ensemble v2 models...
✓ All models loaded successfully!
  - ConvNeXt Base
  - EfficientNet B0
  - MobileNetV3 Large
  - Swin Transformer Base

✓ Ensemble classifier v2 ready!
  Main function: predict_raw(image)
  Alternatives: predict_raw_weighted(image, weights_dict)
               predict_raw_per_model(image)
               ensemble_predict(image)


### With tracking

In [ ]:
# from classfier import *
# from tracking_utils import TrackerSystem
# image_to_project =  patch = cv2.imread(r'C:\git\PhysicalAdverserialProj\results\best_patch_16x16_1_2025-12-13_16_22.png')

# tracker = TrackerSystem(system, predict_raw, weights, printed_aruco_id=10)
# caps, results = tracker.track_project_and_classify(image_to_project)

## Create GIF

In [ ]:
import imageio

captures = caps
# Create GIF from captures
captures_rgb = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in captures]
output_gif_path = 'tracked_jeep_16x16_1_2025-12-13_16_22_inception_v3.gif'
imageio.mimsave(output_gif_path, captures_rgb, fps=14, loop=0)
print(f"GIF saved to {output_gif_path} with {len(captures)} frames")